<a href="https://colab.research.google.com/github/Syed-Waleed-Hussain/Flyrank-ML-Internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

My Data Contract (Lane 2: Content Refresh Scoring):

Grain (What one row means): One row equals one unique page/content item per day (Client X Content X Day).

Tables used: fact_content_daily_performance and dim_content from the FlyRank HF warehouse.

Time window: I am using a mid-panel month, specifically March 2026 (2026-03), to strictly avoid touching the final sealed test month (June).

Target / Proxy: is_declining (derived from trend_direction == 'down').

Deliberately excluded: I am completely excluding outcome-based flags like health_score or action_type because predicting them using themselves would cause data leakage.

In [ ]:
import duckdb
import pandas as pd
from google.colab import userdata

# Load HF token securely
hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")

fact_daily = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')"
print("Connection established successfully!")

Connection established successfully!


## 2. Fields: feature / label / context / excluded
*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Target / Proxy: has_organic_traffic (derived from sessions_organic > 0).
Deliberately excluded: I am completely excluding the raw sessions_organic count from the honest feature set because using it to predict itself causes data leakage.

**My Five Features:**
1. `gsc_impressions`: Knowable at the decision moment because it strictly counts historical Search Console data.
2. `sessions_organic`: Knowable at the decision moment because it relies on past Google Analytics organic traffic logs.
3. `sessions_direct`: Knowable at the decision moment because it relies on past direct traffic logs.
4. `sessions_paid`: Knowable at the decision moment because it relies on past paid traffic logs.
5. `sessions_ai`: Knowable at the decision moment because it relies on past AI-driven traffic logs.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
import numpy as np

df_trap = con.sql(f"""
    SELECT gsc_impressions, sessions_organic, sessions_direct, sessions_paid, sessions_ai
    FROM {fact_daily}
    WHERE report_date = '2026-03-15'
    LIMIT 5000
""").df()

df_trap.fillna(0, inplace=True)

y = (df_trap['sessions_organic'] > 0).astype(int)

X_honest = df_trap[['gsc_impressions', 'sessions_direct', 'sessions_paid', 'sessions_ai']]

X_cheating = X_honest.copy()
X_cheating['leaked_feature'] = df_trap['sessions_organic']

# Training both models
X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(X_honest, y, test_size=0.2, random_state=42)
model_honest = RandomForestClassifier(random_state=42, max_depth=3).fit(X_train_h, y_train_h)

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_cheating, y, test_size=0.2, random_state=42)
model_cheating = RandomForestClassifier(random_state=42, max_depth=3).fit(X_train_c, y_train_c)

print(f"Honest Model Accuracy: {accuracy_score(y_test_h, model_honest.predict(X_test_h)):.2f}")
print(f"TRAP Model Accuracy: {accuracy_score(y_test_c, model_cheating.predict(X_test_c)):.2f} (Perfect score due to Data Leakage!)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Honest Model Accuracy: 1.00
TRAP Model Accuracy: 1.00 (Perfect score due to Data Leakage!)


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
print("--- Query 1: The Grain (Checking uniqueness) ---")
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) as row_count
    FROM {fact_daily}
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING row_count > 1
""").df()
print(f"Duplicates found: {len(grain_check)} (Expected: 0)")

print("\n--- Query 2: Slice Row Count & Date Span ---")
span_check = con.sql(f"""
    SELECT COUNT(*) as total_rows, MIN(report_date) as start_date, MAX(report_date) as end_date
    FROM {fact_daily}
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
""").df()
print(span_check.to_string(index=False))

print("\n--- Query 3: Availability (IS TRUE) ---")
availability_check = con.sql(f"""
    SELECT COUNT(*) as valid_rows
    FROM {fact_daily}
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
    AND ga4_data_available IS TRUE
""").df()
print(f"Rows with GA4 data available: {availability_check['valid_rows'][0]:,}")

--- Query 1: The Grain (Checking uniqueness) ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicates found: 0 (Expected: 0)

--- Query 2: Slice Row Count & Date Span ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 total_rows start_date   end_date
    9841378 2026-03-01 2026-03-31

--- Query 3: Availability (IS TRUE) ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows with GA4 data available: 413,966



## 4. Data limits
*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Limitation of this slice:**
A major limitation is that different client web properties have different tracking installation dates. A page might show `0` for sessions in 2026-03 not because it had no traffic, but because the GA4 tracking simply hadn't been set up yet. Filtering rigorously with `IS TRUE` is mandatory before training.

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.